In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 문장 -> 임베딩 벡터(1차원숫자배열)
- openAI의 text-embedding-3-large(text-embedding-3-small)모델 : .env에 OPENAI_API_KEY 추가
- upstage(https://console.upstage.ai)의 solar-embedding-2-query 모델 : .env에 UPSTAGE_API_KEY추가

# 1. 환경변수 load

In [4]:
from dotenv import load_dotenv
import os
load_dotenv(
    #dotenv_path='.env'
)
openai_key = os.getenv('OPENAI_API_KEY')
upstage_key = os.getenv('UPSTAGE_API_KEY')
print(openai_key[:5])
print(upstage_key[:5])

sk-pr
up_Jc


# 2. 유사도 계산하는 방법
- 1. 유클리드 거리 : 두 벡터간 거리가 가까운지
- 2. 코사인 유사도 : 두 벡터간 방향이 유사한지
- 3. dot product : 두 벡터간 곱을 사용하여 거리와 방향을 모두 고려

In [6]:
import numpy as np
def cosine_similarity(vec1, vec2):
    '두 벡터의 코사인 유사도를 반환'
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0
    return dot_product / (norm_vec1*norm_vec2)

In [8]:
v1 = np.array([3, 4])
v2 = np.array([3, 4.1])
v3 = np.array([25, -9])
print(cosine_similarity(v1, v2))
print(cosine_similarity(v1, v3))

0.9999302571572745
0.2935568725030428


# 3. openAI의 embedding model 사용
- text-embedding-3-large(text-embedding-3-small)모델

In [9]:
from openai import OpenAI
openai_client = OpenAI(
    # api_key=openai_key
)

In [27]:
# text-embedding-3-small(과금) : 1536차원 / text-embedding-3-large:3072차원
response = openai_client.embeddings.create(
    input = "The king is the prince's father",
    model='text-embedding-3-large'
)

In [28]:
# response
print(response.usage)

Usage(prompt_tokens=7, total_tokens=7)


In [29]:
king_vector = np.array(response.data[0].embedding) # The king is the prince's father문장을 숫자로 임베딩벡터
print(king_vector.shape)
print(king_vector)

(3072,)
[ 0.03369141  0.03378296 -0.00437927 ...  0.01652527  0.01048279
 -0.005867  ]


In [30]:
# 과금
response = openai_client.embeddings.create(
    input = "The queen is the prince's mother",
    model='text-embedding-3-large'  
)

In [32]:
queen_vector = np.array(response.data[0].embedding)
print(queen_vector.shape)
print(queen_vector)

(3072,)
[ 0.01913452  0.00117111 -0.00397873 ...  0.00573349  0.00817871
  0.01078796]


In [33]:
# king문장과 queen문장의 유사도
print('king문장과 queen문장의 코사인 유사도 :',
     cosine_similarity(king_vector, queen_vector))

king문장과 queen문장의 코사인 유사도 : 0.6953830686006353


In [34]:
# 과금
response = openai_client.embeddings.create(
    input = "The slave begs",
    model='text-embedding-3-large'  
)

In [35]:
slave_vector = np.array(response.data[0].embedding)
print(slave_vector.shape)
print(slave_vector)

(3072,)
[-0.00867462  0.00682449  0.0075531  ... -0.00827789 -0.01128387
  0.00193214]


In [36]:
print('king문장과 slave문장의 유사도',
     cosine_similarity(king_vector, slave_vector))

king문장과 slave문자의 유사도 0.18330119848430385


In [37]:
print('queen문장과 slave문장의 유사도',
     cosine_similarity(queen_vector, slave_vector))

queen문장과 slave문장의 유사도 0.14570572841094076


In [ ]:
# 한국어 문장을 벡터로 바꿔도 유사도가 비슷?

In [38]:
# 과금
response = openai_client.embeddings.create(
    input = "왕은 왕자의 아버지다",
    model='text-embedding-3-large'  
)

In [39]:
kor_king_vector = np.array(response.data[0].embedding)
print(kor_king_vector.shape)
print(kor_king_vector)

(3072,)
[0.01495361 0.00810242 0.00315094 ... 0.00747299 0.01152039 0.00862885]


In [ ]:
# 과금
response = openai_client.embeddings.create(
    input = "여왕은 왕자의 어머니다",
    model='text-embedding-3-large'  
)